In [ ]:
import gzip
import json
import pickle

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from IPython.display import VimeoVideo
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier


In [ ]:
VimeoVideo("694058667", h="44426f200b", width=600)


In [ ]:
VimeoVideo("694058628", h="00b4cfd027", width=600)


In [ ]:
def wrangle(filename):
    
    # Open compressed file, load into dictionary
    with gzip.open(filename, "r") as f:
        data = json.load(f)
    
    # Load dictionary into DataFrame, set index
    df = pd.DataFrame().from_dict(data["data"]).set_index("company_id")
   
    return df


In [ ]:
df = wrangle("data/poland-bankruptcy-data-2009.json.gz")
print(df.shape)
df.head()


In [ ]:
VimeoVideo("694058591", h="8fc20629aa", width=600)


In [ ]:
# Inspect DataFrame
df.shape
df.info()
df.head(5)


In [ ]:
VimeoVideo("694058537", h="01caf9ae83", width=600)


In [ ]:
# Plot class balance
df["bankrupt"].value_counts(normalize=True).plot(
    kind="bar",
    xlabel="Bankrupt",
    ylabel="Frequency",
    title="Class Balance"
)


In [ ]:
VimeoVideo("694058487", h="6e066151d9", width=600)


In [ ]:
# Create boxplot
sns.boxplot(x="bankrupt", y="feat_27", data=df)
plt.xlabel("Bankrupt")
plt.ylabel("POA / financial expenses")
plt.title("Distribution of Profit/Expenses Ratio, by Class");


In [ ]:
VimeoVideo("694058435", h="8f0ae805d6", width=600)


In [ ]:
# Summary statistics for `feat_27`
df['feat_27'].describe().apply("{0:.0f}".format)


In [ ]:
VimeoVideo("694058398", h="1078bb6d8b", width=600)


In [ ]:
# Plot histogram of `feat_27`
df['feat_27'].hist()

plt.xlabel("POA / financial expenses")
plt.ylabel("Count"),
plt.title("Distribution of Profit/Expenses Ratio");


In [ ]:
VimeoVideo("694058328", h="4aecdc442d", width=600)


In [ ]:
# Create clipped boxplot
q1, q9 = df["feat_27"].quantile([0.1, 0.9])
mask = df["feat_27"].between(q1, q9)

sns.boxplot(x="bankrupt", y="feat_27", data=df[mask])
plt.xlabel("Bankrupt")
plt.ylabel("POA / financial expenses")
plt.title("Distribution of Profit/Expenses Ratio, by Bankruptcy Status")


In [ ]:
# Explore another feature
sns.boxplot(x="bankrupt", y="feat_28", data=df[mask])
plt.xlabel("Bankrupt")
plt.ylabel("POA / financial expenses")
plt.title("Distribution of working capital / fixed assets ratio, by Bankruptcy Status")


In [ ]:
VimeoVideo("694058273", h="85b3be2f63", width=600)


In [ ]:
corr = df.drop(columns='bankrupt').corr()
sns.heatmap(corr)


In [ ]:
target = "bankrupt"
X = df.drop(columns=[target])
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


In [ ]:
VimeoVideo("694058220", h="00c3a98358", width=600)


In [ ]:
under_sampler = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = under_sampler.fit_resample(X_train, y_train)
print(X_train_under.shape)
X_train_under.head()


In [ ]:
VimeoVideo("694058177", h="5cef977f2d", width=600)


In [ ]:
over_sampler = RandomOverSampler(random_state=42)
X_train_over, y_train_over = over_sampler.fit_resample(X_train, y_train)
print(X_train_over.shape)
X_train_over.head()


In [ ]:
VimeoVideo("694058140", h="7ae111412f", width=600)


In [ ]:
acc_baseline = y_train.value_counts(normalize=True).max()
print("Baseline Accuracy:", round(acc_baseline, 4))


In [ ]:
VimeoVideo("694058110", h="dc751751bf", width=600)


In [ ]:
# Fit on `X_train`, `y_train`

model_reg = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(random_state=42)
)
model_reg.fit(X_train, y_train)  

# Fit on `X_train_under`, `y_train_under`

model_under = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(random_state=42)
)
model_under.fit(X_train_under, y_train_under)

# Fit on `X_train_over`, `y_train_over`

model_over = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(random_state=42)
)
model_over.fit(X_train_over, y_train_over)  


In [ ]:
VimeoVideo("694058076", h="d57fb27d07", width=600)


In [ ]:
for m in [model_reg, model_under, model_over]:
    acc_train = m.score(X_train, y_train)
    acc_test = m.score(X_test, y_test)

    print("Training Accuracy:", round(acc_train, 4))
    print("Test Accuracy:", round(acc_test, 4))


In [ ]:
acc_train


In [ ]:
acc_test


In [ ]:
VimeoVideo("694058022", h="ce29f57dee", width=600)


In [ ]:
y_test.value_counts()


In [ ]:
# Plot confusion matrix
ConfusionMatrixDisplay.from_estimator(model_reg, X_test, y_test)


In [ ]:
VimeoVideo("694057996", h="73882663cf", width=600)


In [ ]:
depth = model_over.named_steps["decisiontreeclassifier"].get_depth()
print(depth)


In [ ]:
VimeoVideo("694057962", h="f60aa3b614", width=600)


In [ ]:
# Get importances
importances = model_over.named_steps["decisiontreeclassifier"].feature_importances_

# Put importances into a Series
feat_imp = pd.Series(importances, index=X_train_over.columns).sort_values()
# Plot series
feat_imp.tail(15).plot(kind="barh")
plt.xlabel("Gini Importance")
plt.ylabel("Feature")
plt.title("model_over Feature Importance");


In [ ]:
VimeoVideo("694057923", h="85a50bb588", width=600)


In [ ]:
# Save your model as `"model-5-2.pkl"`
with open("model-5-2.pkl", "wb") as f:
    pickle.dump(model_over, f)


In [ ]:
VimeoVideo("694057859", h="fecd8f9e54", width=600)


In [ ]:
# Load `"model-5-2.pkl"`
with open("model-5-2.pkl", "rb") as f:
    loaded_model = pickle.load(f)
print(loaded_model)
